In [1]:
import pandas as pd
books_df = pd.read_csv(r"C:\Users\LOQ\project_books\books_enriched.csv")

In [2]:
books_df = books_df.drop(columns=["average_rating", "ratings_count"])

In [3]:
print(books_df.columns)

Index(['google_id', 'title', 'subtitle', 'authors', 'publisher',
       'published_date', 'description', 'categories', 'page_count', 'language',
       'isbn_10', 'isbn_13', 'thumbnail', 'search_category',
       'openlibrary_description', 'openlibrary_subjects', 'match_method',
       'openlibrary_categories', 'categories_final', 'categories_sbert',
       'description_final'],
      dtype='str')


In [4]:
for col in [
    "title",
    "subtitle",
    "authors",
    "categories",
    "description"
]:
    books_df[col] = (
        books_df[col]
        .fillna("")
        .str.strip()
    )

In [5]:
books_df = books_df[
    books_df["language"].fillna("").str.lower().eq("en")
].copy()

books_df.reset_index(drop=True, inplace=True)

In [6]:
import re
import unicodedata

def clean_author(author):
    if pd.isna(author):
        return ""

    author = unicodedata.normalize("NFKC", str(author))
    author = author.lower()
    # Remove punctuation except commas, which separate authors
    author = re.sub(r"[^\w\s,]", "", author, flags=re.UNICODE)
    # Normalize spaces without removing author separators
    author = re.sub(r"\s+", " ", author).strip()
    # Clean spaces around commas
    author = re.sub(r"\s*,\s*", ", ", author)
    return author

def normalize_title(title):
    title = str(title)
    title = unicodedata.normalize("NFKC", title)
    title = title.lower()
    title = re.sub(r"\([^)]*\)", "", title)
    title = re.sub(r"[^\w\s]", " ", title, flags=re.UNICODE)
    title = re.sub(r"\s+", " ", title).strip()
    return title

def clean_category(category):
    category = str(category)
    category = unicodedata.normalize("NFKC", category)
    category = category.lower().strip()
    category = re.sub(
        r"[^\w\s,/-]",
        " ",
        category,
        flags=re.UNICODE
    )
    category = re.sub(r"\s+", " ", category).strip()
    return category


def clean_description(description):
    if pd.isna(description):
        return ""
    description = str(description)
    description = unicodedata.normalize("NFKC", description)
    description = re.sub(r"<[^>]+>", " ", description)
    description = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        description
    )
    description = re.sub(r"\s+", " ", description)
    description = "".join(
        char for char in description
        if unicodedata.category(char) != "Cc"
        or char in "\n\t"
    )
    description = re.sub(r"\s+", " ", description).strip()
    return description

In [7]:
books_df["author_clean"] = books_df["authors"].apply(clean_author)
books_df["title_clean"] = books_df["title"].apply(normalize_title)
books_df["category_clean"] = books_df["categories_sbert"].apply(clean_category)
books_df["clean_description"] = books_df["description_final"].apply(clean_description)

In [8]:
duplicates = books_df[
    books_df["title_clean"].duplicated(keep=False)
].sort_values("title_clean")

print(
    duplicates[
        ["title_clean", "authors", "google_id", "isbn_10", "isbn_13"]
    ].to_string(index=False)
)

                                                                                    title_clean                                                                                                                                     authors    google_id    isbn_10      isbn_13
                                                                                           1984                                                                                                            George Orwell, Vlada Stoiljković FrELJwAACAAJ 9958383020 9.789958e+12
                                                                                           1984                                                                                                                               George Orwell 5aPi0QEACAAJ 9394941150 9.789395e+12
                                                                                           1984                                                                                      

In [9]:
# Identifying spelling Variations
all_authors = set()

for authors in books_df["authors"].dropna():
    for author in str(authors).split(","):
        author = clean_author(author)
        if author:
            all_authors.add(author)

print("Unique author strings:", len(all_authors))

Unique author strings: 4641


In [10]:
from collections import defaultdict

authors_by_surname = defaultdict(list)

for author in all_authors:
    surname = author.split()[-1]
    authors_by_surname[surname].append(author)

In [11]:
from rapidfuzz import fuzz
possible_author_aliases = []

for surname, authors in authors_by_surname.items():

    if len(authors) < 2:
        continue

    for i in range(len(authors)):
        for j in range(i + 1, len(authors)):

            a = authors[i]
            b = authors[j]

            score = fuzz.token_set_ratio(a, b)

            if score >= 70:
                possible_author_aliases.append({
                    "surname": surname,
                    "author1": a,
                    "author2": b,
                    "score": score
                })

In [12]:
author_alias_df = pd.DataFrame(possible_author_aliases)

author_alias_df.sort_values(
    "score",
    ascending=False
).head(10)

,surname,author1,author2,score
710,philbrick,w rodman philbrick,rodman philbrick,100.0
681,joyce,james james joyce,james joyce,100.0
52,smith,wilbur smith,wilbur a smith,100.0
1170,bacon,francis bacon,sir francis bacon,100.0
1173,virgil,virgil virgil,virgil,100.0
1174,kabir,saint kabir,kabir,100.0
36,smith,former associate professor in the faculty of m...,barry smith,100.0
1176,michael,anne michael,michael,100.0
722,lahaye,tim f lahaye,tim lahaye,100.0
636,fitzgerald,francis scott key fitzgerald,francis scott fitzgerald,100.0


In [ ]:
HONORIFICS = {
    "mr",
    "mrs",
    "ms",
    "miss",
    "dr",
    "prof",
    "professor",
    "sir",
    "lady",
    "lord",
    "canon",
    "count",
    "rev",
    "reverend",
    "dame",
    "bishop"
}

SUFFIXES = {
    "jr",
    "sr",
    "ii",
    "iii",
    "iv",
    "v",
    "md",
    "phd",
    "esq",
}

In [14]:
def remove_author_titles(author):
    words = clean_author(author).split()

    words = [
        word for word in words
        if word not in HONORIFICS
        and word not in SUFFIXES
    ]

    return " ".join(words)

In [15]:
def author_name_parts(author):
    author = remove_author_titles(author)

    if not author:
        return []

    return author.split()

In [16]:
def authors_likely_same(author1, author2):
    a = author_name_parts(author1)
    b = author_name_parts(author2)

    if not a or not b:
        return False

    # Require the same surname
    if a[-1] != b[-1]:
        return False

    first_a = "".join(a[:-1])
    first_b = "".join(b[:-1])

    # Exact match
    if first_a == first_b:
        return True

    # Compare initials against full names Example: "jrr" vs "johnronaldreuel"
    if first_a and first_b:
        if len(first_a) <= len(first_b):
            shorter, longer = first_a, first_b
        else:
            shorter, longer = first_b, first_a
        if len(shorter) == 1:
            return longer.startswith(shorter)

        # Combined initials match the first letters of full-name components
        shorter_parts = a[:-1] if len(first_a) <= len(first_b) else b[:-1]
        longer_parts = b[:-1] if len(first_a) <= len(first_b) else a[:-1]

        if all(len(part) == 1 for part in shorter_parts):
            if len(shorter_parts) <= len(longer_parts):
                return all(
                    initial == full_name[0]
                    for initial, full_name in zip(
                        shorter_parts, longer_parts
                    )
                )

        # Combined initials, e.g. "jrr", against full-name initials
        initials = "".join(part[0] for part in longer_parts if part)
        if shorter == initials[:len(shorter)]:
            return True

    return False

In [17]:
author_groups = []
assigned = set()

for surname, authors in authors_by_surname.items():
    for author in authors:
        if author in assigned:
            continue

        group = {author}
        for other in authors:
            if other == author or other in assigned:
                continue

            if authors_likely_same(author, other):
                group.add(other)

        assigned.update(group)
        author_groups.append(group)

In [18]:
canonical_map = {}
for group in author_groups:
    canonical_name = min(group, key=len)

    for author in group:
        canonical_map[author] = canonical_name

In [19]:
for author, canonical in list(canonical_map.items()):
    if author != canonical:
        print(f"{author}  ->  {canonical}")

sir william smith  ->  william smith
lisa j smith  ->  l j smith
joanne k rowling  ->  jk rowling
j k rowling  ->  jk rowling
joanne kathleen rowling  ->  jk rowling
canon ronald brownrigg  ->  ronald brownrigg
jacqueline susann  ->  j susann
l m l m montgomery  ->  l m montgomery
m t anderson  ->  mt anderson
jonathan locke hart  ->  j v hart
elwyn brooks white  ->  e b white
virginia c andrews  ->  vc andrews
clive staples lewis  ->  c s lewis
mohandas k gandhi  ->  m k gandhi
john ronald reuel tolkien  ->  jrr tolkien
j r r tolkien  ->  jrr tolkien
t j klune  ->  tj klune
h rider h rider haggard  ->  h r haggard
sir h rider haggard  ->  h rider haggard
james matthew barrie  ->  j m barrie
sir j m barrie  ->  j m barrie
herbert george wells  ->  h g wells
e l james  ->  el james
sir arthur conan doyle  ->  arthur conan doyle
ralph waldo emerson  ->  r w emerson
sue townsend  ->  s townsend
gabriella ballarin  ->  g ballarin
gilbert keith chesterton  ->  g k chesterton
gilbert k chest

In [20]:
def get_author_set(author_string):
    if pd.isna(author_string) or not str(author_string).strip():
        return set()

    authors = str(author_string).split(",")
    canonical_authors = set()

    for author in authors:
        author = clean_author(author)

        if not author:
            continue

        canonical_author = canonical_map.get(
            author,
            author
        )

        canonical_authors.add(canonical_author)

    return canonical_authors


books_df["author_set"] = books_df["author_clean"].apply(get_author_set)

In [21]:
books_df.loc[
    books_df["authors"].str.contains(
        "tolkien", case=False, na=False
    ),
    ["title", "author_set"]
].head(20)

,title,author_set
77,The Lord of the Rings,{jrr tolkien}
93,The Hobbit,{jrr tolkien}
95,The Fellowship of the Ring,{jrr tolkien}
142,Roverandom,{jrr tolkien}
153,The Hobbit,{j r tolkien}
879,Tales From The Perilous Realm,{jrr tolkien}
1142,The Fellowship of the Ring,{jrr tolkien}
2207,The Fellowship of the Ring,{jrr tolkien}
3864,"The Hobbit, Or, There and Back Again",{jrr tolkien}
4937,The Hobbit,{jrr tolkien}


In [22]:
books_df.loc[
    books_df["authors"].str.contains(
        "Simone Brioni", na=False
    ),
    ["authors", "author_set"]
].head()

,authors,author_set
11,"Simone Brioni, Daniele Comberiati","{simone brioni, daniele comberiati}"


In [23]:
print(books_df[[
    "authors",
    "author_set"
]].tail(10))

                                   authors  \
6853                            Saket Shah   
6854                            Harper Lee   
6855  Glencoe Publishing Staff, Harper Lee   
6856                      Erskine Caldwell   
6857                         Jack Kirkland   
6858                      Walter R. Turner   
6859      Mikhail Aleksandrovich Sholokhov   
6860      Mikhail Aleksandrovich Sholokhov   
6861                           A. A. Milne   
6862                         Wayne W. Dyer   

                                  author_set  
6853                            {saket shah}  
6854                            {harper lee}  
6855  {harper lee, glencoe publishing staff}  
6856                      {erskine caldwell}  
6857                         {jack kirkland}  
6858                       {walter r turner}  
6859      {mikhail aleksandrovich sholokhov}  
6860      {mikhail aleksandrovich sholokhov}  
6861                             {a a milne}  
6862                   

In [94]:
import re
import unicodedata
import pandas as pd
from rapidfuzz import fuzz, process

TITLE_THRESHOLD = 82
AUTHOR_THRESHOLD = 85
AUTHOR_THRESHOLD2 = 75
MISSING_AUTHOR_TITLE_THRESHOLD = 92

books = books_df.copy().reset_index(drop=True)

books["title_clean"] = books["title_clean"].fillna("").astype(str)
books["author_clean"] = books["author_clean"].fillna("").astype(str)
books["authors"] = books["authors"].fillna("").astype(str)


def normalize_text(text):
    text = unicodedata.normalize("NFKD", str(text).casefold())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", text)).strip()

normalized_titles = books["title_clean"].map(normalize_text).tolist()
candidate_pairs = set()

for index, title in enumerate(normalized_titles):
    if not title:
        continue
    matches = process.extract(
        title,
        normalized_titles[index + 1:],
        scorer=fuzz.ratio,
        score_cutoff=TITLE_THRESHOLD,
        limit=None)
    
    candidate_pairs.update(
        (index, index + 1 + match_index)
        for _, _, match_index in matches)

def author_similarity(set1, set2):
    if not set1 or not set2:
        return None
    names1 = list(set1)
    names2 = list(set2)

    matches = []
    for i, a in enumerate(names1):
        for j, b in enumerate(names2):
            score = fuzz.ratio(a, b)
            if score >= 70:
                matches.append((score, i, j))

    # Greedy one-to-one matching
    matched1, matched2 = set(), set()

    for score, i, j in sorted(matches, reverse=True):
        if i not in matched1 and j not in matched2:
            matched1.add(i)
            matched2.add(j)

    matched = len(matched1)
    # Measures how much the smaller set overlaps the larger one
    return matched / min(len(names1), len(names2)) * 100

possible_duplicates = []
for index1, index2 in sorted(candidate_pairs):
    row1 = books.iloc[index1]
    row2 = books.iloc[index2]
    title1 = normalize_text(row1["title_clean"])
    title2 = normalize_text(row2["title_clean"])
    title_score = fuzz.ratio(title1, title2)

    if title_score < TITLE_THRESHOLD:
        continue

    authors1 = row1["author_set"]
    authors2 = row2["author_set"]
    # Treat missing or title-like author values as unknown
    unknown1 = (not authors1 or normalize_text(row1["author_clean"]) == title1)
    unknown2 = (not authors2 or normalize_text(row2["author_clean"]) == title2)

    if unknown1 or unknown2:
        author_score = None
        is_duplicate = title_score >= MISSING_AUTHOR_TITLE_THRESHOLD

    elif title_score >= 95:
        author_score = author_similarity(authors1, authors2)
        is_duplicate = (author_score is not None
        and author_score >= AUTHOR_THRESHOLD2)

    else:
        author_score = author_similarity(authors1, authors2)
        is_duplicate = (author_score is not None
            and author_score >= AUTHOR_THRESHOLD)

    if is_duplicate:
        possible_duplicates.append({
            "index1": index1,
            "index2": index2,
            "title1": row1["title"],
            "title2": row2["title"],
            "title_score": title_score,
            "authors1": row1["authors"],
            "authors2": row2["authors"],
            "author_set1": authors1,
            "author_set2": authors2,
            "author_score": author_score
        })

possible_duplicates_df = pd.DataFrame(possible_duplicates)

if not possible_duplicates_df.empty:
    possible_duplicates_df = possible_duplicates_df.sort_values(
        ["title_score", "author_score"],
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)

print("Potential duplicate pairs:", len(possible_duplicates_df))
display(possible_duplicates_df.head(50))

Potential duplicate pairs: 48


,index1,index2,title1,title2,title_score,authors1,authors2,author_set1,author_set2,author_score
0,244,4361,Pride and Prejudice,Pride and Prejudice,100.000000,Jane Austen,Austen,{jane austen},{austen},100.0
1,244,4366,Pride and Prejudice,Pride and Prejudice,100.000000,Jane Austen,Jane Jane Austen,{jane austen},{jane jane austen},100.0
2,383,414,Brother Odd,Brother Odd,100.000000,Dean Ray Koontz,Dean Koontz,{dean ray koontz},{dean koontz},100.0
3,814,1668,Pygmalion,Pygmalion (Warbler Classics Annotated Edition),100.000000,Bernard Shaw,George Bernard Shaw,{bernard shaw},{george bernard shaw},100.0
4,830,1613,Saint Joan,Saint Joan,100.000000,Bernard Shaw,George Bernard Shaw,{bernard shaw},{george bernard shaw},100.0
5,855,890,Arms and the Man,Arms and the Man,100.000000,George Shaw,George Bernard Shaw,{george shaw},{george bernard shaw},100.0
6,1526,3774,Odyssey,Odyssey,100.000000,Omero,Homer,{omero},{homer},100.0
7,1615,1668,Pygmalion,Pygmalion (Warbler Classics Annotated Edition),100.000000,George Shaw,George Bernard Shaw,{george shaw},{george bernard shaw},100.0
8,1637,1712,An Enemy of the People,An Enemy of the People,100.000000,Henrik Johan Ibsen,Henrik Ibsen,{henrik johan ibsen},{henrik ibsen},100.0
9,1882,5232,Wuthering Heights,Wuthering Heights,100.000000,Emily Brontë,Emily Jane Bronte,{emily brontë},{emily jane bronte},100.0


In [95]:
duplicates_df = pd.DataFrame(possible_duplicates)

duplicates_df

,index1,index2,title1,title2,title_score,authors1,authors2,author_set1,author_set2,author_score
0,244,4361,Pride and Prejudice,Pride and Prejudice,100.000000,Jane Austen,Austen,{jane austen},{austen},100.0
1,244,4366,Pride and Prejudice,Pride and Prejudice,100.000000,Jane Austen,Jane Jane Austen,{jane austen},{jane jane austen},100.0
2,383,414,Brother Odd,Brother Odd,100.000000,Dean Ray Koontz,Dean Koontz,{dean ray koontz},{dean koontz},100.0
3,814,1668,Pygmalion,Pygmalion (Warbler Classics Annotated Edition),100.000000,Bernard Shaw,George Bernard Shaw,{bernard shaw},{george bernard shaw},100.0
4,830,1613,Saint Joan,Saint Joan,100.000000,Bernard Shaw,George Bernard Shaw,{bernard shaw},{george bernard shaw},100.0
5,855,890,Arms and the Man,Arms and the Man,100.000000,George Shaw,George Bernard Shaw,{george shaw},{george bernard shaw},100.0
6,981,1602,On the Nature of Things,The Nature of Things,93.023256,Titus Lucretius Carus,Lucretius Lucretius,{titus lucretius carus},{lucretius lucretius},100.0
7,989,4454,Alice in Wonderland and Through the Looking Glass,Alice's Adventures in Wonderland and Through t...,88.288288,Lewis Lewis Carroll,Lewis Carroll,{lewis lewis carroll},{lewis carroll},100.0
8,1526,3774,Odyssey,Odyssey,100.000000,Omero,Homer,{omero},{homer},100.0
9,1615,1668,Pygmalion,Pygmalion (Warbler Classics Annotated Edition),100.000000,George Shaw,George Bernard Shaw,{george shaw},{george bernard shaw},100.0


In [96]:
important_columns = [
    "authors",
    "publisher",
    "published_date",
    "description",
    "categories",
    "isbn_10",
    "isbn_13",
]

# Make empty strings and whitespace-only strings count as missing
for col in important_columns:
    books_df[col] = books_df[col].replace(
        r"^\s*$", pd.NA, regex=True
    )
books_df["completeness"] = (
    books_df[important_columns]
    .notna()
    .sum(axis=1)
)

In [97]:
rows_to_drop = set()

for _, row in duplicates_df.iterrows():

    index1 = row["index1"]
    index2 = row["index2"]

    score1 = books_df.loc[index1, "completeness"]
    score2 = books_df.loc[index2, "completeness"]

    if books_df.loc[index1, "authors"] == "" or books_df.loc[index2, "authors"] == "":
        if books_df.loc[index1, "authors"] == "":
            rows_to_drop.add(index1)
        else:
            rows_to_drop.add(index2)

    elif books_df.loc[index1, "categories"] == "" or books_df.loc[index2, "categories"] == "":
        if books_df.loc[index1, "categories"] == "":
            rows_to_drop.add(index1)
        else:
            rows_to_drop.add(index2)

    elif books_df.loc[index1, "description"] == "" or books_df.loc[index2, "description"] == "":
        if books_df.loc[index1, "description"] == "":
            rows_to_drop.add(index1)
        else:
            rows_to_drop.add(index2)
     
    else:
        if score1 >= score2:
            rows_to_drop.add(index2)
        else:
            rows_to_drop.add(index1)

In [98]:
print("books to drop:" , len(rows_to_drop))

books to drop: 46


In [99]:
books_df = books_df.drop(
    index=list(rows_to_drop)
).reset_index(drop=True)

In [100]:
books_df.loc[
    books_df["title"].str.contains(
        "harry potter", case=False, na=False
    ),
    ["title", "author_clean"]
].head(20)

,title,author_clean
77,Harry Potter and the Sorcerer's Stone,j k rowling
920,Harry Potter and the Chamber of Secrets,j k rowling
1024,Harry Potter and the Prisoner of Azkaban,j k rowling
1052,Harry Potter and the Half-Blood Prince,j k rowling
1924,Harry Potter and the Deathly Hallows,j k rowling
1932,Harry Potter and the Order of the Phoenix,j k rowling
1935,Harry Potter and the Goblet of Fire,jk rowling
4325,Harry Potter and the Deathly Hallows - Gryffin...,j k rowling
4326,Harry Potter and the Half-Blood Prince - Huffl...,j k rowling
4327,Harry Potter,"j k rowling, j k rowling, fantastic stories"


In [101]:
books_df.loc[
    books_df["title"].str.contains(
        "rebecca", case=False, na=False
    ),
    ["title",  "authors", "author_set"]
].head(20) 

,title,authors,author_set
4778,The Key to Rebecca,Ken Follett,{ken follett}
4779,New Chronicles of Rebecca,Kate Douglas Smith Wiggin,{kate douglas smith wiggin}
4780,Rebecca,Daphne du Mourier,{daphne du mourier}
4781,Rebecca West: Artist and Thinker,Peter Wolfe,{peter wolfe}
4782,Rebecca of Sunnybrook Farm Illustrated,Kate Douglas Wiggin,{kate douglas wiggin}


In [124]:
NON_BOOK_TERMS = [
    "notebook",
    "workbook",
    "study guide",
    "student workbook",
    "teacher guide",
    "teacher's guide",
    "answer key",
    "summary",
    "study notes",
    "companion guide",
    "lesson plan",
    "test prep",
    "exam prep",
    "journal",
    "coloring book",
    "activity book",
    "trivia book",
    "a guide to",
    "movie tie in",
    "movie guide",
    "book analysis",
    "trivia on books",
    "bright summaries"
]


In [125]:
# Normalize all non-book terms
NON_BOOK_TERMS = [
    normalize_title(term)
    for term in NON_BOOK_TERMS
]

def is_non_book(title):
    normalized_title = normalize_title(title)

    return any(
        term in normalized_title
        for term in NON_BOOK_TERMS
        if term
    )

In [126]:
books_df = books_df[
    ~books_df["title_clean"].apply(is_non_book)
].copy()

books_df = books_df[
    ~books_df["author_clean"].apply(is_non_book)
].copy()

books_df.reset_index(drop=True, inplace=True)

In [127]:
books_df.head(10)

,google_id,title,subtitle,authors,publisher,published_date,description,categories,page_count,language,...,openlibrary_categories,categories_final,categories_sbert,description_final,author_clean,title_clean,category_clean,clean_description,author_set,completeness
0,_DYNAQAAMAAJ,The Face of Another,,Kōbō Abe,Kodansha,1992,The Japanese novelist Kobo Abe has often been ...,Fiction,260.0,en,...,Near and far eastern fiction (fictional works ...,"Fiction, Near and far eastern fiction (fiction...",Near and far eastern fiction (fictional works ...,The Japanese novelist Kobo Abe has often been ...,kōbō abe,the face of another,near and far eastern fiction fictional works b...,The Japanese novelist Kobo Abe has often been ...,{kōbō abe},5
1,ggpgAAAAMAAJ,Generations of Winter,,Василий Аксенов,Random House (NY),1994,"""Written in the great tradition of epic Russia...",Fiction,616.0,en,...,NaN,Fiction,Fiction,"""Written in the great tradition of epic Russia...",василий аксенов,generations of winter,fiction,"""Written in the great tradition of epic Russia...",{василий аксенов},5
2,v7VPEQAAQBAJ,The Street of Crocodiles,,Bruno Schulz,Rare Treasure Editions,2025-03-19T00:00:00Z,''The Street of Crocodiles'' by Bruno Schulz (...,Fiction,136.0,en,...,"Fiction, Polish fiction, Fantasy, Polish Short...","Fiction, Polish fiction, Fantasy, Polish Short...","Polish fiction, Fantasy, Polish Short stories,...",''The Street of Crocodiles'' by Bruno Schulz (...,bruno schulz,the street of crocodiles,"polish fiction, fantasy, polish short stories,...",''The Street of Crocodiles'' by Bruno Schulz (...,{bruno schulz},7
3,FSTRAgAAQBAJ,Kokoro,Hints and Echoes of Japanese Inner Life,Lafcadio Hearn,Tuttle Publishing,2012-12-20,The 15 classic essays collected in Kokoro exam...,Literary Collections,267.0,en,...,NaN,Literary Collections,Literary Collections,The 15 classic essays collected in Kokoro exam...,lafcadio hearn,kokoro,literary collections,The 15 classic essays collected in Kokoro exam...,{lafcadio hearn},7
4,qvk6DwAAQBAJ,Sense and Sensibility - Illustrated Edition,,Jane Austen,Abela Publishing Ltd,2017-10-23,SENSE AND SENSIBILITY written by Jane Austen w...,Fiction,579.0,en,...,NaN,Fiction,Fiction,SENSE AND SENSIBILITY written by Jane Austen w...,jane austen,sense and sensibility illustrated edition,fiction,SENSE AND SENSIBILITY written by Jane Austen w...,{jane austen},7
5,XUtNEQAAQBAJ,The Final Problem,,Arthur Conan Doyle,SAMPI Books,2025-03-08,"In ""The Final Problem"", Sherlock Holmes finds ...",Fiction,39.0,en,...,NaN,Fiction,Fiction,"In ""The Final Problem"", Sherlock Holmes finds ...",arthur conan doyle,the final problem,fiction,"In ""The Final Problem"", Sherlock Holmes finds ...",{arthur conan doyle},7
6,l4MZAAAAYAAJ,A House-boat on the Styx,,John Kendrick Bangs,NaN,1898,The premise of the book is that everyone who h...,Fiction,238.0,en,...,"Fiction, Houseboats, Fiction, fantasy, histori...","Fiction, Houseboats, fantasy, historical, coll...","Houseboats, fantasy, historical, collections &...",The premise of the book is that everyone who h...,john kendrick bangs,a house boat on the styx,"houseboats, fantasy, historical, collections a...",The premise of the book is that everyone who h...,{john kendrick bangs},4
7,Rf2dEAAAQBAJ,Virgin Soil,in large print,Ivan Sergeevich Turgenev,BoD – Books on Demand,2022-11-25,Reproduction of the original.,Fiction,429.0,en,...,NaN,Fiction,Fiction,Reproduction of the original.,ivan sergeevich turgenev,virgin soil,fiction,Reproduction of the original.,{ivan sergeevich turgenev},7
8,69JaAAAAMAAJ,Hemingway,A Life Without Consequences,James R. Mellow,NaN,1992,"Summary: A biography of the legendary writer, ...",Biography & Autobiography,742.0,en,...,"American Novelists, Biography, Hemingway, Erne...","Biography & Autobiography, American Novelists,...","Biography & Autobiography, American Novelists,...","Summary: A biography of the legendary writer, ...",james r mellow,hemingway,"biography autobiography, american novelists, 

In [128]:
books_df.loc[
    books_df["title_clean"].str.contains(
        "(Book Analysis)", case=False, na=False
    ),
    ["title_clean", "author_clean"]
].head(20)

C:\Users\LOQ\AppData\Local\Temp\ipykernel_10544\1073689159.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  books_df["title_clean"].str.contains(


,title_clean,author_clean


In [129]:
books_df.to_csv(
    "processed_books_dataset_enriched.csv",
    index=False
)

print("Saved:", len(books_df), "books")

Saved: 5266 books
